In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Also add parent directories of the current directory (when running from notebooks/)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 03. Introduction to Calculus and Optimization - From Rate of Change to Gradient Descent

> **Learning Goals**
> - Understand derivatives as "instantaneous rates of change" and "slopes of tangent lines."
> - Verify how gradients $\nabla f$ point in the direction of steepest increase.
> - Connect gradient descent to neural network training.

## 3.1 Derivatives

The derivative $f'(x)$ measures the **instantaneous rate of change** of a function $f$.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h)-f(x)}{h}$$

Geometrically, it is the **slope of the tangent line** at the point $(x, f(x))$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Numerical differentiation implementation
def numerical_diff(f, x, h=1e-5):
    """Numerical derivative using the central difference method."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Example function
def f(x):
    return x**2

def f_prime(x):
    return 2 * x  # Analytic derivative

# Comparison
x0 = 3.0
print(f"f(x) = x^2, x = {x0}")
print(f"Analytic derivative f'(x) = 2x = {f_prime(x0)}")
print(f"Numerical derivative (h=1e-5): {numerical_diff(f, x0):.6f}")
print(f"Error: {abs(numerical_diff(f, x0) - f_prime(x0)):.2e}")


In [ ]:
# Visualize the geometric meaning of derivatives
def f(x): return x**2

x = np.linspace(-2, 4, 200)
y = f(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# function and tangent line
ax = axes[0]
ax.plot(x, y, 'b-', linewidth=2, label='$f(x) = x^2$')
x0 = 2.0
y0 = f(x0)
slope = 2 * x0
tangent = slope * (x - x0) + y0
ax.plot(x, tangent, 'r--', linewidth=2, label=f'tangent line (gradient={slope:.1f})')
ax.plot(x0, y0, 'ro', markersize=10)
ax.annotate(f'Tangent point ({x0}, {y0})', xy=(x0, y0), xytext=(x0+0.3, y0-2),
            fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Derivative = Slope of Tangent Line')
ax.legend(); ax.grid(True, alpha=0.3)

# Numerical differentiation error by h
ax = axes[1]
hs = np.logspace(-16, 0, 50)
errors = [abs(numerical_diff(f, 1.0, h=h) - 2.0) for h in hs]
ax.loglog(hs, errors, 'o-', markersize=4)
ax.set_xlabel('h')
ax.set_ylabel('Absolute error')
ax.set_title('Numerical differentiation error vs h (central difference)')
ax.grid(True, alpha=0.3, which='both')
# If h is too small, floating-point error occurs; if too large, truncation error occurs
ax.axvline(1e-5, color='r', linestyle='--', alpha=0.5, label='Optimal h ≈ 1e-5')
ax.legend()

plt.tight_layout()
plt.savefig('../figures/ch03_derivative.png', dpi=100, bbox_inches='tight')
plt.show()
print("Warning: if h is too small, floating-point error occurs; if h is too large, truncation error occurs.")


## 3.2 Differentiation Rules and the Chain Rule

Common rules:

- $(x^n)' = nx^{n-1}$
- $(f+g)' = f' + g'$
- $(fg)' = f'g + fg'$

**Chain rule**: for a composite function,

$$\frac{d}{dx} f(g(x)) = f'(g(x))g'(x)$$

The chain rule is the mathematical core of backpropagation. We revisit it in Chapter 07.


In [ ]:
# Numerical verification of the chain rule
def f(g): return g**2          # f(g) = g^2
def g(x): return np.sin(x)    # g(x) = sin(x)
# Composition: f(g(x)) = sin(x)^2

# Numerical derivative of the composed function directly
def compose(x): return f(g(x))
df_dx_numerical = numerical_diff(compose, 1.0)

# Analytic derivative (chain rule): f'(g(x)) * g'(x) = 2*sin(x) * cos(x)
x0 = 1.0
df_dx_chain = 2 * np.sin(x0) * np.cos(x0)

print(f"f(g(x)) = sin(x)^2, x = {x0}")
print(f"Numerical derivative: {df_dx_numerical:.6f}")
print(f"Chain rule: 2*sin(x)*cos(x) = {df_dx_chain:.6f}")
print(f"Error: {abs(df_dx_numerical - df_dx_chain):.2e}")


## 3.3 Multivariable Functions and Partial Derivatives

For $f(x, y)$, the partial derivative $\frac{\partial f}{\partial x}$ measures the rate of change with respect to $x$ while holding $y$ fixed.

**Gradient** is the vector of all partial derivatives:

$$\nabla f = \left[\frac{\partial f}{\partial x_1}, \ldots, \frac{\partial f}{\partial x_n}\right]$$

The gradient points in the **steepest ascent direction** of the function. The opposite direction, $-\nabla f$, is the steepest descent direction.


In [ ]:
# Gradient of a multivariable function
def f_2d(x, y):
    return x**2 + 3*y**2  # Elliptical contours

def grad_f_2d(x, y):
    return np.array([2*x, 6*y])  # [df/dx, df/dy]

# Numerical gradient
def numerical_grad_2d(f, x, y, h=1e-5):
    dfdx = (f(x+h, y) - f(x-h, y)) / (2*h)
    dfdy = (f(x, y+h) - f(x, y-h)) / (2*h)
    return np.array([dfdx, dfdy])

x0, y0 = 1.5, 1.0
print(f"f(x, y) = x^2 + 3y^2, (x, y) = ({x0}, {y0})")
print(f"Analytic gradient: {grad_f_2d(x0, y0)}")
print(f"Numerical gradient:   {numerical_grad_2d(f_2d, x0, y0)}")

# Visualize contours and gradient directions
fig, ax = plt.subplots(figsize=(8, 7))
xs = np.linspace(-2, 2, 100)
ys = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(xs, ys)
Z = X**2 + 3*Y**2
cs = ax.contour(X, Y, Z, levels=20, cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8)

# Gradient arrows at multiple points
for xi in np.linspace(-1.5, 1.5, 7):
    for yi in np.linspace(-1.5, 1.5, 7):
        g = grad_f_2d(xi, yi)
        # Normalize to display direction only
        g_norm = g / (np.linalg.norm(g) + 1e-8) * 0.2
        ax.arrow(xi, yi, g_norm[0], g_norm[1], head_width=0.05,
                 head_length=0.03, fc='red', ec='red', alpha=0.7)

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('gradient $\\nabla f$ direction (red arrows)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch03_gradient_field.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> The gradient is always perpendicular to contour lines and points in the steepest ascent direction.")


## 3.4 Gradient Descent

Optimization problem: $\min_{\mathbf{x}} f(\mathbf{x})$

Update rule:

$$\mathbf{x}_{t+1} = \mathbf{x}_t - \eta \nabla f(\mathbf{x}_t)$$

- $\eta$: learning rate. If it is too small, learning is slow; if it is too large, training can diverge.
- Move in the **negative gradient direction**, the direction of steepest descent.


In [ ]:
# gradient descent implementation
def gradient_descent_2d(f, grad_f, x0, y0, lr=0.1, n_steps=50):
    """Run gradient descent on a 2D function."""
    path = [(x0, y0)]
    x, y = x0, y0
    for _ in range(n_steps):
        gx, gy = grad_f(x, y)
        x = x - lr * gx
        y = y - lr * gy
        path.append((x, y))
    return np.array(path)

# f(x, y) = x^2 + 3y^2 (minimum point: (0, 0))
path = gradient_descent_2d(f_2d, grad_f_2d, x0=1.8, y0=1.5, lr=0.1, n_steps=50)
print(f"Starting point: ({path[0, 0]:.3f}, {path[0, 1]:.3f})")
print(f"After 50 steps: ({path[-1, 0]:.6f}, {path[-1, 1]:.6f})")
print(f"Minimum value f(0, 0) = 0, reached value f = {f_2d(path[-1, 0], path[-1, 1]):.2e}")

# Visualization
from llm_math.viz import plot_contour_and_path
fig, ax = plt.subplots(figsize=(8, 7))
plot_contour_and_path(
    ax, lambda v: f_2d(v[0], v[1]), path,
    xlim=(-2, 2), ylim=(-2, 2),
    title='Gradient descent path (lr=0.1)'
)
plt.tight_layout()
plt.savefig('../figures/ch03_gd_path.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Compare convergence for different learning rates
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lr in zip(axes, [0.01, 0.1, 0.3]):
    path = gradient_descent_2d(f_2d, grad_f_2d, x0=1.8, y0=1.5, lr=lr, n_steps=50)
    plot_contour_and_path(
        ax, lambda v: f_2d(v[0], v[1]), path,
        xlim=(-2, 2), ylim=(-2, 2),
        title=f'lr={lr} (final f={f_2d(*path[-1]):.2e})'
    )
plt.tight_layout()
plt.savefig('../figures/ch03_lr_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> If the learning rate is small, convergence is slow; if it is too large, training diverges or oscillates.")


## 3.5 Convex vs. Non-Convex Optimization

**Convex function**: every local minimum is also a global minimum. Optimization is relatively easy.
- Examples: $f(x)=x^2$, linear regression MSE loss

**Non-convex function**: many local minima and saddle points can exist.
- Neural network loss functions are non-convex.
- In high dimensions, saddle points are often a more important issue than local minima.


In [ ]:
# Convex vs non-convex function examples
def convex_f(x, y): return x**2 + y**2
def nonconvex_f(x, y): return np.sin(x) * np.cos(y) + 0.1 * (x**2 + y**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xs = np.linspace(-3, 3, 100)
ys = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(xs, ys)

# Convex
Z1 = convex_f(X, Y)
cs = axes[0].contour(X, Y, Z1, levels=20, cmap='viridis')
axes[0].clabel(cs, inline=True, fontsize=8)
axes[0].set_title('Convex: f(x,y) = x² + y² (one global minimum)')
axes[0].set_aspect('equal')

# Non-convex
Z2 = nonconvex_f(X, Y)
cs = axes[1].contour(X, Y, Z2, levels=20, cmap='viridis')
axes[1].clabel(cs, inline=True, fontsize=8)
axes[1].set_title('Non-convex: f(x,y) = sin(x)cos(y) + 0.1(x²+y²)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.savefig('../figures/ch03_convex_nonconvex.png', dpi=100, bbox_inches='tight')
plt.show()
print("Neural-network loss functions are high-dimensional non-convex functions. Optimization is difficult, but works surprisingly well in practice.")
print("Reason: in high dimensions, saddle points are far more common than local minima, and saddle points can often be escaped.")


## 3.6 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Derivative | $f'(x)=\lim_{h\to0}\frac{f(x+h)-f(x)}{h}$ | Instantaneous rate of change |
| Chain rule | $\frac{d}{dx}f(g(x))=f'(g)g'(x)$ | Derivative of a composite function, core of backpropagation |
| Partial derivative | $\frac{\partial f}{\partial x_i}$ | Rate of change along one variable |
| Gradient | $\nabla f=[\partial f/\partial x_i]$ | Steepest ascent direction |
| Gradient descent | $\mathbf{x}\leftarrow\mathbf{x}-\eta\nabla f$ | Optimization update |

### Practice Problems
1. Derive the derivative of $f(x)=x^3-3x^2+2$ and verify it with numerical differentiation.
2. Derive $\nabla f$ analytically for $f(x,y)=x^2y+\sin(xy)$ and verify it numerically.
3. Run gradient descent on $f(x,y)=(x-3)^2+(y+2)^2$ and compare convergence speed for different learning rates.
4. Explain what happens when the learning rate is too large.
5. Visualize the saddle point of $f(x,y)=x^2-y^2$ and discuss why optimization is difficult near it.
